In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.sparse import load_npz
from sklearn.metrics.pairwise import cosine_similarity

DATA_DIR = Path(r"../datasets")
PROCESSED_DIR = DATA_DIR / "processed"

train = pd.read_csv(PROCESSED_DIR / "ratings_train.csv")
test = pd.read_csv(PROCESSED_DIR / "ratings_test.csv")
movies = pd.read_csv(PROCESSED_DIR / "phase2_movies.csv")
weighted_matrix = load_npz(PROCESSED_DIR / "weighted_matrix.npz")

user_ids = np.load(PROCESSED_DIR / "cf_user_ids.npy")
movie_ids = np.load(PROCESSED_DIR / "cf_movie_ids.npy")
user_factors = np.load(PROCESSED_DIR / "cf_user_factors.npy")
movie_factors = np.load(PROCESSED_DIR / "cf_movie_factors.npy")

user_to_index = pd.Series(np.arange(len(user_ids)), index=user_ids)
movie_id_to_index = pd.Series(movies.index.to_numpy(), index=movies["movieId"])
from scipy.sparse import csr_matrix

weighted_matrix = csr_matrix(weighted_matrix)

print("Weighted matrix:", weighted_matrix.shape)
print("Type:", type(weighted_matrix))

POSITIVE_THRESHOLD = 4.0


In [ ]:
positive_test = test[
    test["rating"] >= POSITIVE_THRESHOLD
]

relevant_by_user = (
    positive_test
    .groupby("userId")["movieId"]
    .apply(set)
    .to_dict()
)

evaluation_users = list(relevant_by_user)

print("Evaluation users:", len(evaluation_users))


In [11]:
def minmax(values):
    values = np.asarray(values, dtype=float)
    lo = np.nanmin(values)
    hi = np.nanmax(values)
    if hi == lo:
        return np.zeros_like(values)
    return (values - lo) / (hi - lo)

def content_scores(user_id):

    history = train[
        (train["userId"] == user_id) &
        (train["rating"] >= POSITIVE_THRESHOLD)
    ].copy()

    history["idx"] = history["movieId"].map(
        movie_id_to_index
    )

    history = history.dropna(
        subset=["idx"]
    )

    if history.empty:
        return None

    indices = (
        history["idx"]
        .astype(int)
        .to_numpy()
    )

    vectors = weighted_matrix[indices]

    weights = (
        history["rating"]
        .to_numpy(dtype=float) - 3.0
    )

    user_vector = (
        vectors.multiply(weights[:, None])
        .sum(axis=0)
        / weights.sum()
    )

    user_vector = csr_matrix(user_vector)

    return cosine_similarity(
        user_vector,
        weighted_matrix
    ).ravel()

def collaborative_scores(user_id):
    if user_id not in user_to_index.index:
        return None

    uidx = int(user_to_index[user_id])
    raw = user_factors[uidx] @ movie_factors.T

    series = pd.Series(raw, index=movie_ids)

    return movies["movieId"].map(series).fillna(0).to_numpy()

def ranked_recommendations(
    user_id,
    model="content",
    k=10,
    content_weight=0.5
):
    c = content_scores(user_id)

    if c is None:
        return pd.DataFrame()

    c = minmax(c)

    if model == "content":
        score = c

    elif model == "collaborative":
        cf = collaborative_scores(user_id)
        if cf is None:
            return pd.DataFrame()
        score = minmax(cf)

    elif model == "hybrid":
        cf = collaborative_scores(user_id)
        if cf is None:
            return pd.DataFrame()
        score = (
            content_weight * c +
            (1 - content_weight) * minmax(cf)
        )

    else:
        raise ValueError("Unknown model.")

    result = movies.copy()
    result["score"] = score

    rated = set(
        train.loc[
            train["userId"] == user_id,
            "movieId"
        ]
    )

    result = result[
        ~result["movieId"].isin(rated)
    ]

    return (
        result
        .sort_values("score", ascending=False)
        .head(k)
    )


In [12]:
def evaluate_model(model, k=10, content_weight=0.5):
    rows = []

    for user_id, relevant in relevant_by_user.items():

        recs = ranked_recommendations(
            user_id=user_id,
            model=model,
            k=k,
            content_weight=content_weight
        )

        if recs.empty:
            continue

        hits = len(
            set(recs["movieId"]) & relevant
        )

        rows.append({
            "userId": user_id,
            "precision": hits / k,
            "recall": hits / len(relevant),
            "hit_rate": int(hits > 0)
        })

    if not rows:
        return None

    result = pd.DataFrame(rows)

    return {
        "model": model,
        "k": k,
        "precision_at_k": result["precision"].mean(),
        "recall_at_k": result["recall"].mean(),
        "hit_rate_at_k": result["hit_rate"].mean(),
        "users": len(result)
    }


In [ ]:
baseline_rows = []

for model in ["content", "collaborative"]:
    for k in [5, 10]:
        result = evaluate_model(model, k)
        if result:
            baseline_rows.append(result)

baseline_results = pd.DataFrame(baseline_rows)
display(baseline_results)


In [ ]:
weight_grid = [
    0.00,
    0.10,
    0.20,
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
    0.80,
    0.90,
    1.00
]

hybrid_rows = []

for weight in weight_grid:
    result = evaluate_model(
        model="hybrid",
        k=10,
        content_weight=weight
    )

    if result:
        result["content_weight"] = weight
        result["collaborative_weight"] = 1 - weight
        hybrid_rows.append(result)

hybrid_results = pd.DataFrame(hybrid_rows)

display(
    hybrid_results.sort_values(
        "hit_rate_at_k",
        ascending=False
    )
)


In [ ]:
best_hybrid = (
    hybrid_results
    .sort_values(
        ["hit_rate_at_k", "precision_at_k"],
        ascending=False
    )
    .iloc[0]
)

print("Best hybrid configuration:")
display(best_hybrid.to_frame("value"))


In [ ]:
evaluation_dir = DATA_DIR / "evaluation"
evaluation_dir.mkdir(exist_ok=True)

baseline_results.to_csv(
    evaluation_dir / "baseline_results.csv",
    index=False
)

hybrid_results.to_csv(
    evaluation_dir / "hybrid_weight_results.csv",
    index=False
)

print("Evaluation results saved.")


In [ ]:
assert not baseline_results.empty
assert not hybrid_results.empty
assert 0 <= float(best_hybrid["content_weight"]) <= 1

print("PHASE 7 EVALUATION AND OPTIMIZATION VALIDATION PASSED")
